# 07 — Hyperparameter Tuning

**Goal**: Optimize the tree-based models (LightGBM and XGBoost) on the handcrafted feature set using Optuna, maximizing the cluster-stratified AUC-ROC.

**Verify gates**:
- Tune LightGBM for 30 trails
- Tune XGBoost for 30 trails
- Get final OOF (out-of-fold) predictions with tuned parameters
- Save tuned OOF predictions for Phase 8 (Ensembling)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING) # Keep logs clean
plt.style.use('dark_background')
sns.set_palette('viridis')
print('Libraries loaded ✓')

Libraries loaded ✓


## Step 7.1 — Load Features & Clusters

In [2]:
X_train = pd.read_pickle('../data/processed/X_train_features.pkl')
train_df = pd.read_csv('../data/processed/train_clusters.csv')
y_train = train_df['Label'].values
cluster_labels = train_df['Cluster'].values

print(f'Train shape: {X_train.shape}')

Train shape: (7143, 827)


## Step 7.2 — Tuning LightGBM

In [3]:
def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    
    cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
    auc_scores = []
    
    # We use 3 splits to speed up tuning
    for train_idx, val_idx in cv.split(X_train, y_train, groups=cluster_labels):
        X_tr, y_tr = X_train.iloc[train_idx], y_train[train_idx]
        X_va, y_va = X_train.iloc[val_idx], y_train[val_idx]
        
        model = LGBMClassifier(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict_proba(X_va)[:, 1]
        auc_scores.append(roc_auc_score(y_va, preds))
        
    return np.mean(auc_scores)

print('Tuning LightGBM...')
study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lgb, n_trials=30, n_jobs=1, show_progress_bar=True)

print(f'Best LightGBM AUC: {study_lgb.best_value:.4f}')
print(f'Best Params: {study_lgb.best_params}')

Tuning LightGBM...


  0%|          | 0/30 [00:00<?, ?it/s]

Best LightGBM AUC: 0.8911
Best Params: {'n_estimators': 199, 'learning_rate': 0.04254680102198071, 'max_depth': 9, 'num_leaves': 65, 'subsample': 0.775074435411378, 'colsample_bytree': 0.5064958421380898, 'min_child_samples': 11}


## Step 7.3 — Tuning XGBoost

In [4]:
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'eval_metric': 'auc',
        'random_state': 42,
        'n_jobs': -1
    }

    # Note: Label encoding inside objective to ensure clean mapping from -1/1 to 0/1 for xgb
    y_mapped = np.where(y_train == 1, 1, 0)

    cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
    auc_scores = []
    
    for train_idx, val_idx in cv.split(X_train, y_mapped, groups=cluster_labels):
        X_tr, y_tr = X_train.iloc[train_idx], y_mapped[train_idx]
        X_va, y_va = X_train.iloc[val_idx], y_mapped[val_idx]
        
        model = XGBClassifier(**params)
        model.fit(X_tr, y_tr)
        preds = model.predict_proba(X_va)[:, 1]
        auc_scores.append(roc_auc_score(y_va, preds))
        
    return np.mean(auc_scores)

print('Tuning XGBoost...')
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=30, n_jobs=1, show_progress_bar=True)

print(f'Best XGBoost AUC: {study_xgb.best_value:.4f}')
print(f'Best Params: {study_xgb.best_params}')

Tuning XGBoost...


  0%|          | 0/30 [00:00<?, ?it/s]

Best XGBoost AUC: 0.8912
Best Params: {'n_estimators': 239, 'learning_rate': 0.09855278327575058, 'max_depth': 8, 'subsample': 0.8882848681492092, 'colsample_bytree': 0.6688641226046078, 'min_child_weight': 1}


## Step 7.4 — Get Full 5-Fold OOF Predictions with Best Params

In [5]:
def get_oof_predictions(model_class, best_params, is_xgb=False):
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(X_train))
    
    y_mapped = np.where(y_train == 1, 1, 0) if is_xgb else y_train

    params = best_params.copy()
    params['random_state'] = 42
    params['n_jobs'] = -1
    if not is_xgb:
        params['verbose'] = -1
    else:
        params['eval_metric'] = 'auc'

    for train_idx, val_idx in cv.split(X_train, y_mapped, groups=cluster_labels):
        X_tr, y_tr = X_train.iloc[train_idx], y_mapped[train_idx]
        X_va, y_va = X_train.iloc[val_idx], y_mapped[val_idx]
        
        model = model_class(**params)
        model.fit(X_tr, y_tr)
        oof_preds[val_idx] = model.predict_proba(X_va)[:, 1]
        
    final_auc = roc_auc_score(np.where(y_train == 1, 1, 0), oof_preds)
    return oof_preds, final_auc

print('Generating full 5-fold OOF predictions...')
oof_lgb, auc_lgb = get_oof_predictions(LGBMClassifier, study_lgb.best_params, is_xgb=False)
oof_xgb, auc_xgb = get_oof_predictions(XGBClassifier, study_xgb.best_params, is_xgb=True)

print(f'Final 5-Fold LGBM AUC:   {auc_lgb:.4f}')
print(f'Final 5-Fold XGBoost AUC: {auc_xgb:.4f}')


Generating full 5-fold OOF predictions...
Final 5-Fold LGBM AUC:   0.8931
Final 5-Fold XGBoost AUC: 0.8941


In [6]:
oof_df = pd.DataFrame({
    'Sequence': train_df['Sequence'],
    'Label': y_train,
    'tuned_lgb_pred': oof_lgb,
    'tuned_xgb_pred': oof_xgb
})
oof_df.to_csv('../data/processed/oof_tuned_trees.csv', index=False)
print('Saved tuned tree model out-of-fold predictions to data/processed/oof_tuned_trees.csv!')


Saved tuned tree model out-of-fold predictions to data/processed/oof_tuned_trees.csv!
